In [2]:
# ============================================
# DATASET AUDIT - Load Dataset
# ============================================

import os
import numpy as np
import pandas as pd

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("✅ Dataset Loaded!")
print("Train :", X_train.shape, y_train.shape)
print("Val   :", X_val.shape, y_val.shape)
print("Test  :", X_test.shape, y_test.shape)

✅ Dataset Loaded!
Train : (7608, 154, 63) (7608,)
Val   : (275, 154, 63) (275,)
Test  : (487, 154, 63) (487,)


In [3]:
# ============================================
# DATASET AUDIT - Class Distribution
# ============================================

num_classes = 210

train_counts = np.bincount(y_train, minlength=num_classes)
val_counts = np.bincount(y_val, minlength=num_classes)
test_counts = np.bincount(y_test, minlength=num_classes)

distribution = pd.DataFrame({
    "Class": np.arange(num_classes),
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts
})

print("=" * 55)
print("📊 DATASET SUMMARY")
print("=" * 55)

print(f"Classes     : {num_classes}")
print(f"Train       : {len(y_train)}")
print(f"Validation  : {len(y_val)}")
print(f"Test        : {len(y_test)}")

print("\n" + "=" * 55)
print("📌 MIN / MAX SAMPLES PER CLASS")
print("=" * 55)

print(f"Train      -> Min: {train_counts.min()} | Max: {train_counts.max()}")
print(f"Validation -> Min: {val_counts.min()} | Max: {val_counts.max()}")
print(f"Test       -> Min: {test_counts.min()} | Max: {test_counts.max()}")

print("\n" + "=" * 55)
print("🚨 CLASSES WITH 0 VALIDATION SAMPLES")
print("=" * 55)

zero_val = np.where(val_counts == 0)[0]
print(zero_val)
print("Total:", len(zero_val))

print("\n" + "=" * 55)
print("⚠️ CLASSES WITH ONLY 1 VALIDATION SAMPLE")
print("=" * 55)

one_val = np.where(val_counts == 1)[0]
print(one_val)
print("Total:", len(one_val))

distribution.to_csv("../outputs/class_distribution.csv", index=False)

print("\n✅ class_distribution.csv saved to outputs/")

📊 DATASET SUMMARY
Classes     : 210
Train       : 7608
Validation  : 275
Test        : 487

📌 MIN / MAX SAMPLES PER CLASS
Train      -> Min: 18 | Max: 51
Validation -> Min: 1 | Max: 2
Test       -> Min: 1 | Max: 3

🚨 CLASSES WITH 0 VALIDATION SAMPLES
[]
Total: 0

⚠️ CLASSES WITH ONLY 1 VALIDATION SAMPLE
[  0   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  19
  21  22  25  27  29  31  33  35  37  39  41  43  44  47  49  51  53  55
  57  59  60  61  62  63  64  65  67  68  70  72  74  76  78  80  82  84
  86  87  89  91  94  95  96  98  99 100 102 104 106 108 109 110 111 112
 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 129 130 131
 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149
 150 151 153 154 155 157 158 160 161 162 163 165 167 168 170 171 173 174
 176 177 179 180 182 184 186 188 189 191 193 195 197 199 201 203 205 207
 209]
Total: 145

✅ class_distribution.csv saved to outputs/


In [4]:
# ============================================
# DATASET AUDIT - Rebuild Balanced Split
# ============================================

import numpy as np

# Combine everything back together
X_all = np.concatenate([X_train, X_val, X_test], axis=0)
y_all = np.concatenate([y_train, y_val, y_test], axis=0)

X_train_new, y_train_new = [], []
X_val_new, y_val_new = [], []
X_test_new, y_test_new = [], []

num_classes = 210
rng = np.random.default_rng(42)   # reproducible split

for cls in range(num_classes):

    idx = np.where(y_all == cls)[0]
    rng.shuffle(idx)

    # 3 validation samples
    val_idx = idx[:3]

    # 2 test samples
    test_idx = idx[3:5]

    # remaining samples go to training
    train_idx = idx[5:]

    X_val_new.append(X_all[val_idx])
    y_val_new.append(y_all[val_idx])

    X_test_new.append(X_all[test_idx])
    y_test_new.append(y_all[test_idx])

    X_train_new.append(X_all[train_idx])
    y_train_new.append(y_all[train_idx])

# Merge lists into arrays
X_train_new = np.concatenate(X_train_new)
y_train_new = np.concatenate(y_train_new)

X_val_new = np.concatenate(X_val_new)
y_val_new = np.concatenate(y_val_new)

X_test_new = np.concatenate(X_test_new)
y_test_new = np.concatenate(y_test_new)

# Shuffle each split independently
def shuffle_data(X, y):
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

X_train_new, y_train_new = shuffle_data(X_train_new, y_train_new)
X_val_new, y_val_new = shuffle_data(X_val_new, y_val_new)
X_test_new, y_test_new = shuffle_data(X_test_new, y_test_new)

print("✅ New balanced split created!\n")
print("Train :", X_train_new.shape, y_train_new.shape)
print("Val   :", X_val_new.shape, y_val_new.shape)
print("Test  :", X_test_new.shape, y_test_new.shape)

✅ New balanced split created!

Train : (7320, 154, 63) (7320,)
Val   : (630, 154, 63) (630,)
Test  : (420, 154, 63) (420,)


In [5]:
# ============================================
# DATASET AUDIT - Verify Balanced Split
# ============================================

train_counts = np.bincount(y_train_new, minlength=210)
val_counts = np.bincount(y_val_new, minlength=210)
test_counts = np.bincount(y_test_new, minlength=210)

print("Validation unique counts:", np.unique(val_counts))
print("Test unique counts:", np.unique(test_counts))

print("\nClasses with incorrect validation count:",
      np.where(val_counts != 3)[0])

print("Classes with incorrect test count:",
      np.where(test_counts != 2)[0])

print("\n✅ Verification complete!")

Validation unique counts: [3]
Test unique counts: [2]

Classes with incorrect validation count: []
Classes with incorrect test count: []

✅ Verification complete!


In [6]:
# ============================================
# DATASET AUDIT - Save Balanced Split
# ============================================

import os

SAVE_PATH = "../dataset/training"
os.makedirs(SAVE_PATH, exist_ok=True)

np.save(os.path.join(SAVE_PATH, "X_train.npy"), X_train_new)
np.save(os.path.join(SAVE_PATH, "y_train.npy"), y_train_new)

np.save(os.path.join(SAVE_PATH, "X_val.npy"), X_val_new)
np.save(os.path.join(SAVE_PATH, "y_val.npy"), y_val_new)

np.save(os.path.join(SAVE_PATH, "X_test.npy"), X_test_new)
np.save(os.path.join(SAVE_PATH, "y_test.npy"), y_test_new)

print("✅ Balanced dataset saved successfully!")

print("\nFinal Shapes:")
print("Train :", X_train_new.shape, y_train_new.shape)
print("Val   :", X_val_new.shape, y_val_new.shape)
print("Test  :", X_test_new.shape, y_test_new.shape)

✅ Balanced dataset saved successfully!

Final Shapes:
Train : (7320, 154, 63) (7320,)
Val   : (630, 154, 63) (630,)
Test  : (420, 154, 63) (420,)
